In [ ]:
import astropy.units as u
from spectral_cube import SpectralCube
import plotly.graph_objects as go
import numpy as np

# Figure size in pixes
WIDTH = 1100
HEIGHT = 800

In [ ]:
cube = SpectralCube.read("RCW79_CII_20_8_0p5.fits")

In [ ]:
# Region of interest
CENTER_RA = 204.9884403046 * u.deg
CENTER_DEC = -61.6995853135 * u.deg
WIDTH_RA = 8 * u.arcmin
WIDTH_DEC = 9 * u.arcmin
VRAD_MIN = -60e3 * u.m / u.s
VRAD_MAX = -30e3 * u.m / u.s

# Minimal flux values for isosurfaces
MIN_FLUX = 4.
MAX_FLUX = 28.

cube = cube.subcube(
    xlo = CENTER_RA - WIDTH_RA / 2,
    xhi = CENTER_RA + WIDTH_RA / 2,
    ylo = CENTER_DEC - WIDTH_DEC / 2,
    yhi = CENTER_DEC + WIDTH_DEC / 2,
    zlo = VRAD_MIN,
    zhi = VRAD_MAX,
)
    

In [ ]:
# Reduce the number of pixels for the note book
cube = cube[::3, ::3, ::3]

In [ ]:
# Arrange the cube in mesh grids
ra, dec, vrad, flux = (
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
)

for i in range(cube.shape[2]):
    ra[:, :, i] = cube.world[0, 0, i][2].value

for i in range(cube.shape[1]):
    dec[:, i, :] = cube.world[0, i, 0][1].value

for i in range(cube.shape[0]):
    vrad[i, :, :] = cube.world[i, 0, 0][0].value

flux = cube.unmasked_data[:]

# Swap RA and Dec axis for the plot.
ra = np.swapaxes(ra, 1, 2)
dec = np.swapaxes(dec, 1, 2)
vrad = np.swapaxes(vrad, 1, 2)
flux = np.swapaxes(flux, 1, 2)


In [ ]:
fig = go.Figure(data=go.Volume(
    x=vrad.flatten(),
    y=ra.flatten(),
    z=dec.flatten(),
    value=flux.flatten(),
    isomin=MIN_FLUX,
    isomax=MAX_FLUX,
    opacity=0.05, # needs to be small to see through all surfaces
    surface_count=30, # needs to be a large number for good volume rendering
    colorscale='viridis',
    caps= dict(x_show=False, y_show=False, z_show=False), # no caps
    )
)

fig.update_layout(
        scene = {
            'xaxis_title': 'vrad [m/s]',
            'yaxis_title': 'RA [deg]',
            'zaxis_title': 'Dec [deg]',
            'aspectmode': 'manual',
            'aspectratio': {
                'x': cube.shape[0],
                'y': cube.shape[1],
                'z': cube.shape[2],
            },
        },
        scene_camera={
            'center': {'x': 0, 'y': 0, 'z': -.1},
        },
        width=WIDTH,
        height=HEIGHT,
        margin={'r': 10, 'b': 10, 'l': 10, 't': 10},
)

fig.show()